# LeetCode #1210: Minimum Moves to Reach Target with Rotations

https://leetcode.com/problems/minimum-moves-to-reach-target-with-rotations/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force (Recursive DFS)** | $O(n^2 \cdot 2)$ | $O(n^2)$ |
| **Optimal: BFS on State Space ★** | $O(n^2)$ | $O(n^2)$ |

---

## Understanding the Methods

### Brute Force (Recursive DFS)
Recursively try all moves (right, down, clockwise rotate, counter-clockwise rotate) with memoization on $(tail\_row, tail\_col, orientation)$. Correct but may revisit states unless carefully pruned.

### Optimal: BFS on State Space ★
Model each configuration as a triple $(row, col, orientation)$ where $(row, col)$ is the tail cell and orientation is 0 (horizontal) or 1 (vertical). BFS explores configurations level by level, guaranteeing the first time the target configuration is reached is the minimum number of moves.

**Why this is better than Brute Force:** BFS naturally gives the shortest path without maintaining explicit distance arrays or back-tracking; visited states prevent cycles.

**Constraints:**
* $2 \le n \le 100$
* $grid[i][j] \in \{0, 1\}$
* Grid is guaranteed solvable or $-1$ is returned

## Solutions

### C#

In [ ]:
using System.Collections.Generic;
public class Solution {
    public int MinimumMoves(int[][] grid) {
        int n = grid.Length;
        // State: (tail_row, tail_col, orientation) where 0=horizontal, 1=vertical
        var visited = new bool[n, n, 2];
        var queue = new Queue<(int r, int c, int orient, int moves)>();
        queue.Enqueue((0, 1, 0, 0)); // snake initially at (0,0)-(0,1), tail at col 1, horizontal
        visited[0, 1, 0] = true;
        while (queue.Count > 0) {
            var (r, c, orient, moves) = queue.Dequeue();
            // Target: horizontal snake with tail at (n-1, n-1)
            if (r == n - 1 && c == n - 1 && orient == 0) return moves;
            // Move right (horizontal snake)
            if (orient == 0 && c + 1 < n && grid[r][c + 1] == 0) {
                if (!visited[r, c + 1, 0]) { visited[r, c + 1, 0] = true; queue.Enqueue((r, c + 1, 0, moves + 1)); }
            }
            // Move down (vertical snake)
            if (orient == 1 && r + 1 < n && grid[r + 1][c] == 0) {
                if (!visited[r + 1, c, 1]) { visited[r + 1, c, 1] = true; queue.Enqueue((r + 1, c, 1, moves + 1)); }
            }
            // Move down (horizontal snake)
            if (orient == 0 && r + 1 < n && grid[r + 1][c] == 0 && grid[r + 1][c - 1] == 0) {
                if (!visited[r + 1, c, 0]) { visited[r + 1, c, 0] = true; queue.Enqueue((r + 1, c, 0, moves + 1)); }
            }
            // Move right (vertical snake)
            if (orient == 1 && c + 1 < n && grid[r][c + 1] == 0 && grid[r - 1][c + 1] == 0) {
                if (!visited[r, c + 1, 1]) { visited[r, c + 1, 1] = true; queue.Enqueue((r, c + 1, 1, moves + 1)); }
            }
            // Rotate clockwise (horizontal -> vertical)
            if (orient == 0 && r + 1 < n && grid[r + 1][c] == 0 && grid[r + 1][c - 1] == 0) {
                if (!visited[r + 1, c, 1]) { visited[r + 1, c, 1] = true; queue.Enqueue((r + 1, c, 1, moves + 1)); }
            }
            // Rotate counter-clockwise (vertical -> horizontal)
            if (orient == 1 && c + 1 < n && grid[r][c + 1] == 0 && grid[r - 1][c + 1] == 0) {
                if (!visited[r, c + 1, 0]) { visited[r, c + 1, 0] = true; queue.Enqueue((r, c + 1, 0, moves + 1)); }
            }
        }
        return -1;
    }
}

### Python

In [ ]:
from collections import deque
class Solution:
    def minimum_moves(self, grid: list[list[int]]) -> int:
        n = len(grid)
        # State: (tail_row, tail_col, orientation) where 0=horizontal, 1=vertical
        visited = set()
        # Snake initially at (0,0)-(0,1), tail at col 1, horizontal
        queue = deque([(0, 1, 0, 0)])
        visited.add((0, 1, 0))
        while queue:
            r, c, orient, moves = queue.popleft()
            # Target: horizontal snake with tail at (n-1, n-1)
            if r == n-1 and c == n-1 and orient == 0:
                return moves
            def enqueue(nr, nc, no):
                if (nr, nc, no) not in visited:
                    visited.add((nr, nc, no)); queue.append((nr, nc, no, moves+1))
            if orient == 0:  # horizontal
                if c+1 < n and grid[r][c+1] == 0: enqueue(r, c+1, 0)     # slide right
                if r+1 < n and grid[r+1][c] == 0 and grid[r+1][c-1] == 0:
                    enqueue(r+1, c, 0)   # slide down
                    enqueue(r+1, c, 1)   # rotate clockwise
            else:  # vertical
                if r+1 < n and grid[r+1][c] == 0: enqueue(r+1, c, 1)     # slide down
                if c+1 < n and grid[r][c+1] == 0 and grid[r-1][c+1] == 0:
                    enqueue(r, c+1, 1)   # slide right
                    enqueue(r, c+1, 0)   # rotate counter-clockwise
        return -1

### Go

In [ ]:
func minimumMoves(grid [][]int) int {
    n := len(grid)
    type State struct{ r, c, orient int }
    visited := map[State]bool{{0, 1, 0}: true}
    type Entry struct{ r, c, orient, moves int }
    queue := []Entry{{0, 1, 0, 0}}
    for len(queue) > 0 {
        cur := queue[0]; queue = queue[1:]
        r, c, orient, moves := cur.r, cur.c, cur.orient, cur.moves
        if r == n-1 && c == n-1 && orient == 0 { return moves }
        enqueue := func(nr, nc, no int) {
            s := State{nr, nc, no}
            if !visited[s] { visited[s] = true; queue = append(queue, Entry{nr, nc, no, moves+1}) }
        }
        if orient == 0 {
            if c+1 < n && grid[r][c+1] == 0 { enqueue(r, c+1, 0) }
            if r+1 < n && grid[r+1][c] == 0 && grid[r+1][c-1] == 0 {
                enqueue(r+1, c, 0); enqueue(r+1, c, 1)
            }
        } else {
            if r+1 < n && grid[r+1][c] == 0 { enqueue(r+1, c, 1) }
            if c+1 < n && grid[r][c+1] == 0 && grid[r-1][c+1] == 0 {
                enqueue(r, c+1, 1); enqueue(r, c+1, 0)
            }
        }
    }
    return -1
}

### Rust

In [ ]:
use std::collections::{VecDeque, HashSet};
impl Solution {
    pub fn minimum_moves(grid: Vec<Vec<i32>>) -> i32 {
        let n = grid.len();
        // State: (tail_row, tail_col, orientation) where 0=horizontal, 1=vertical
        let mut visited = HashSet::new();
        visited.insert((0usize, 1usize, 0u8));
        let mut queue = VecDeque::new();
        queue.push_back((0usize, 1usize, 0u8, 0i32));
        while let Some((r, c, orient, moves)) = queue.pop_front() {
            if r == n-1 && c == n-1 && orient == 0 { return moves; }
            let mut try_add = |nr: usize, nc: usize, no: u8| {
                if !visited.contains(&(nr, nc, no)) {
                    visited.insert((nr, nc, no));
                    queue.push_back((nr, nc, no, moves+1));
                }
            };
            if orient == 0 {
                if c+1 < n && grid[r][c+1] == 0 { try_add(r, c+1, 0); }
                if r+1 < n && grid[r+1][c] == 0 && grid[r+1][c-1] == 0 {
                    try_add(r+1, c, 0); try_add(r+1, c, 1);
                }
            } else {
                if r+1 < n && grid[r+1][c] == 0 { try_add(r+1, c, 1); }
                if c+1 < n && grid[r][c+1] == 0 && grid[r-1][c+1] == 0 {
                    try_add(r, c+1, 1); try_add(r, c+1, 0);
                }
            }
        }
        -1
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `grid = [[0,0,0,0,0,1],[1,1,0,0,1,0],[0,0,0,0,1,1],[0,0,1,0,1,0],[0,1,1,0,0,0],[0,1,1,0,0,0]]`
BFS finds a sequence of right/down/rotate moves totaling **11** moves — it explores all valid states level by level and returns the first time the target is reached.

### 2. Slightly Complex
**Input:** `grid = [[0,0,0],[1,1,0],[0,0,0]]`
The snake can reach $(2,2)$ horizontal in **1** move (slide right once, then down). BFS correctly identifies the shortest path. Answer: **1**.

### 3. Edge Case: Time Factor
**Input:** Fully open $100 \times 100$ grid (all zeros).
BFS processes at most $n^2 \times 2 = 20{,}000$ states; each state is enqueued and dequeued exactly once, keeping runtime tightly at $O(n^2)$.

### 4. Edge Case: Space Factor
**Input:** $100 \times 100$ grid with a maze requiring the snake to traverse every cell.
The visited set and queue together hold $O(n^2)$ states simultaneously; this is the dominant space cost.

### 5. Almost-Impossible but Plausible
**Input:** `grid = [[0,0],[0,0]]` ($n=2$)
Snake starts at $(0,0)-(0,1)$ and target is $(1,1)-(1,0)$ horizontal. It can slide down (not possible if blocked) or rotate. In an open $2\times2$ grid the answer is **3** moves.